<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/modelTrain_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#═══════════════════════════════════════════
# STEP 1: Install required packages
#═══════════════════════════════════════════
import os
os.environ["WANDB_DISABLED"] = "true"

!pip uninstall transformers accelerate peft -y -q
!pip install transformers==4.40.0 accelerate==0.27.2 peft==0.9.0 -q

print("✅ Installation complete!")
print("   → Runtime → Restart Session → Ok")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
✅ Installation complete!
   → Runtime → Restart Session → Ok


In [2]:
#═══════════════════════════════════════════
# STEP 2: Verify GPU + versions
#═══════════════════════════════════════════
import torch, transformers, accelerate

print(f"GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name      : {torch.cuda.get_device_name(0)}")

print(f"transformers  : {transformers.__version__}")
print(f"accelerate    : {accelerate.__version__}")

if not torch.cuda.is_available():
    print("\n⚠️  No GPU! Runtime → Change Runtime Type → T4 GPU")
else:
    print("\n✅ Ready!")


GPU Available : True
GPU Name      : Tesla T4
transformers  : 4.40.0
accelerate    : 0.27.2

✅ Ready!


In [3]:
#═══════════════════════════════════════════
# STEP 3: Upload CSV files
#═══════════════════════════════════════════
import os, pandas as pd
from google.colab import files

os.environ["WANDB_DISABLED"] = "true"

print("Upload 4 files:")
print("  1. train_combined.csv")
print("  2. val_combined.csv")
print("  3. test_combined.csv")
print("  4. unlabeled_dataset.csv")
print("")

uploaded = files.upload()

train_df     = pd.read_csv("train_combined.csv")
val_df       = pd.read_csv("val_combined.csv")
test_df      = pd.read_csv("test_combined.csv")
df_unlabeled = pd.read_csv("unlabeled_dataset.csv",
                           header=None, names=['text', 'source'])

print(f"\n✅ All files loaded!")
print(f"   train_df     : {len(train_df):,} rows")
print(f"   val_df       : {len(val_df):,} rows")
print(f"   test_df      : {len(test_df):,} rows")
print(f"   df_unlabeled : {len(df_unlabeled):,} requirements")
print(f"\nPriority Distribution (train):")
print(train_df['priority'].value_counts())


Upload 4 files:
  1. train_combined.csv
  2. val_combined.csv
  3. test_combined.csv
  4. unlabeled_dataset.csv



Saving unlabeled_dataset.csv to unlabeled_dataset.csv
Saving test_combined.csv to test_combined.csv
Saving val_combined.csv to val_combined.csv
Saving train_combined.csv to train_combined.csv

✅ All files loaded!
   train_df     : 23,274 rows
   val_df       : 2,909 rows
   test_df      : 2,910 rows
   df_unlabeled : 98 requirements

Priority Distribution (train):
priority
Medium    9430
High      8550
Low       5294
Name: count, dtype: int64


In [4]:
#═══════════════════════════════════════════
# STEP 4: Compute class weights
# Fixes Medium being confused with High
#═══════════════════════════════════════════
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

labels_array = train_df['priority_id'].values
weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1, 2]),
    y            = labels_array
)
class_weights = torch.tensor(weights, dtype=torch.float)

print("Class weights (balanced):")
print(f"  High   (0): {weights[0]:.4f}")
print(f"  Medium (1): {weights[1]:.4f}")
print(f"  Low    (2): {weights[2]:.4f}")

Class weights (balanced):
  High   (0): 0.9074
  Medium (1): 0.8227
  Low    (2): 1.4654


In [5]:
#═══════════════════════════════════════════
# STEP 5: Load TinyBERT model
#═══════════════════════════════════════════
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "huawei-noah/TinyBERT_General_4L_312D"

print(f"Loading TinyBERT...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model     = BertForSequenceClassification.from_pretrained(
                model_name,
                num_labels          = 3,
                hidden_dropout_prob = 0.2,   # was default 0.1
                attention_probs_dropout_prob = 0.2,
            )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ TinyBERT loaded!")
print(f"   Device : {device}")
if not torch.cuda.is_available():
    print("\n⚠️  No GPU! Runtime → Change Runtime Type → GPU")

Loading TinyBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ TinyBERT loaded!
   Device : cuda


In [6]:
#═══════════════════════════════════════════
# STEP 6: Dataset + Weighted Trainer
# Key upgrades:
#   max_length 128 → 192  (more context)
#   Custom loss: class weights + label smoothing
#═══════════════════════════════════════════
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

MAX_LENGTH = 192   # was 128

class RequirementsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['priority_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation     = True,
            padding        = 'max_length',
            max_length     = self.max_len,
            return_tensors = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(self.labels[index], dtype=torch.long)
        }

train_dataset = RequirementsDataset(train_df, tokenizer)
val_dataset   = RequirementsDataset(val_df,   tokenizer)
test_dataset  = RequirementsDataset(test_df,  tokenizer)

# ── Custom Trainer: weighted loss + label smoothing ──
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Metrics: track macro F1 ──
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=1)
    return {
        'accuracy'   : round(accuracy_score(labels, preds), 4),
        'f1_macro'   : round(f1_score(labels, preds, average='macro'),    4),
        'f1_weighted': round(f1_score(labels, preds, average='weighted'), 4),
    }

print(f"✅ Dataset and WeightedTrainer ready!")
print(f"   Train : {len(train_dataset):,}  Val : {len(val_dataset):,}  Test : {len(test_dataset):,}")
print(f"   Max length : {MAX_LENGTH} tokens")

✅ Dataset and WeightedTrainer ready!
   Train : 23,274  Val : 2,909  Test : 2,910
   Max length : 192 tokens


In [7]:
#═══════════════════════════════════════════
# STEP 7: Training configuration
# Key upgrades vs original:
#   epochs        5  → 10
#   learning_rate 2e-5 → 5e-5  (TinyBERT needs higher LR)
#   warmup_ratio  0   → 0.1
#   lr_scheduler  linear → cosine
#   weight_decay  0.01 → 0.01  (keep)
#   max_grad_norm added → 1.0
#   best model by macro F1 (not eval_loss)
#═══════════════════════════════════════════
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                  = './tinybert-priority-v2',
    num_train_epochs            = 10,
    per_device_train_batch_size = 32,   # TinyBERT is small, fits larger batch
    per_device_eval_batch_size  = 64,
    learning_rate               = 5e-5,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_macro',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    fp16                        = True,
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    class_weights   = class_weights,
)

print("✅ Trainer configured!")
print("   Epochs       : 10")
print("   LR           : 5e-5 cosine + warmup")
print("   Batch size   : 32 train / 64 eval")
print("   Loss         : Weighted CrossEntropy + label_smoothing=0.1")
print("   Best model   : macro F1")

✅ Trainer configured!
   Epochs       : 10
   LR           : 5e-5 cosine + warmup
   Batch size   : 32 train / 64 eval
   Loss         : Weighted CrossEntropy + label_smoothing=0.1
   Best model   : macro F1


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [8]:
#═══════════════════════════════════════════
# STEP 8: Fine-tune TinyBERT
# ~15-20 minutes on T4 GPU
#═══════════════════════════════════════════
print("=" * 55)
print("      STARTING TINYBERT FINE-TUNING v2")
print("=" * 55)

trainer.train()

print("\n✅ Training Complete!")

      STARTING TINYBERT FINE-TUNING v2


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.772000,0.778742,0.710200,0.717600,0.708200
2,0.704500,0.723745,0.738400,0.746700,0.736500
3,0.684400,0.704567,0.742200,0.746500,0.739900
4,0.651900,0.720945,0.738700,0.745300,0.737600
5,0.597600,0.723712,0.752800,0.761600,0.753300
6,0.588500,0.705571,0.754900,0.763100,0.754100
7,0.574100,0.718490,0.750100,0.758100,0.750000
8,0.557000,0.746466,0.746300,0.753100,0.744700
9,0.535400,0.747186,0.746600,0.754800,0.746200
10,0.528900,0.746441,0.745600,0.754100,0.745200



✅ Training Complete!


In [9]:
#═══════════════════════════════════════════
# STEP 9: Save model — do immediately!
# Do this RIGHT AWAY before session expires!
#═══════════════════════════════════════════
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Save locally
model.save_pretrained("./tinybert-priority-v2")
tokenizer.save_pretrained("./tinybert-priority-v2")
print("✅ Model saved locally!")

# Save to Google Drive permanently
shutil.copytree(
    "./tinybert-priority-v2",
    "/content/drive/MyDrive/tinybert-priority-v2",
    dirs_exist_ok=True
)
print("✅ Model saved to Google Drive!")
print("   Location: MyDrive/tinybert-priority-v2")
print("   Safe — will NOT be deleted when session ends!")

Mounted at /content/drive
✅ Model saved locally!
✅ Model saved to Google Drive!
   Location: MyDrive/tinybert-priority-v2
   Safe — will NOT be deleted when session ends!


In [10]:
#═══════════════════════════════════════════
# STEP 10: Evaluate on test set
#═══════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['priority_id'].tolist()

print("=" * 60)
print("         FINAL MODEL EVALUATION")
print("=" * 60)
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['High', 'Medium', 'Low'],
    digits=4
))

cm    = confusion_matrix(true_labels, pred_labels)
cm_df = pd.DataFrame(
    cm,
    index   = ['Actual High', 'Actual Medium', 'Actual Low'],
    columns = ['Pred High',   'Pred Medium',   'Pred Low']
)
print("Confusion Matrix:")
print(cm_df)

# ── Improvement comparison ──
from sklearn.metrics import accuracy_score, f1_score

new_acc   = accuracy_score(true_labels, pred_labels)
new_macro = f1_score(true_labels, pred_labels, average='macro')

print("\n" + "=" * 60)
print("  IMPROVEMENT vs v1 (TinyBERT original)")
print("=" * 60)
print(f"  Accuracy  : 0.7600 (v1)  →  {new_acc:.4f} (v2)")
print(f"  Macro F1  : 0.7700 (v1)  →  {new_macro:.4f} (v2)")
print("=" * 60)

         FINAL MODEL EVALUATION
              precision    recall  f1-score   support

        High     0.7055    0.8223    0.7594      1069
      Medium     0.8040    0.6853    0.7399      1179
         Low     0.8088    0.8051    0.8070       662

    accuracy                         0.7629      2910
   macro avg     0.7727    0.7709    0.7688      2910
weighted avg     0.7689    0.7629    0.7623      2910

Confusion Matrix:
               Pred High  Pred Medium  Pred Low
Actual High          879          144        46
Actual Medium        291          808        80
Actual Low            76           53       533

  IMPROVEMENT vs v1 (TinyBERT original)
  Accuracy  : 0.7600 (v1)  →  0.7629 (v2)
  Macro F1  : 0.7700 (v1)  →  0.7688 (v2)


In [11]:
#═══════════════════════════════════════════
# STEP 11: Rank all requirements in order
#═══════════════════════════════════════════
import torch

def predict_priority(text):
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = 192        # must match training max_length
    )
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=1)[0]
    pred_id    = probs.argmax().item()
    confidence = round(probs[pred_id].item() * 100, 2)
    priority_map = {0: 'High', 1: 'Medium', 2: 'Low'}
    return priority_map[pred_id], confidence, probs.tolist()

# Predict for all requirements
print("Ranking all requirements...")
results = []

for i, row in df_unlabeled.iterrows():
    priority, confidence, probs = predict_priority(row['text'])
    results.append({
        'text'        : row['text'],
        'source'      : row['source'],
        'priority'    : priority,
        'confidence'  : confidence,
        'prob_high'   : round(probs[0] * 100, 2),
        'prob_medium' : round(probs[1] * 100, 2),
        'prob_low'    : round(probs[2] * 100, 2),
    })

df_results = pd.DataFrame(results)

# Sort: High → Medium → Low, then by confidence descending
priority_order = {'High': 0, 'Medium': 1, 'Low': 2}
df_results['sort_priority']   = df_results['priority'].map(priority_order)
df_results['sort_confidence'] = -df_results['confidence']
df_results = df_results.sort_values(
    ['sort_priority', 'sort_confidence']
).drop(
    columns=['sort_priority', 'sort_confidence']
).reset_index(drop=True)

df_results.index      += 1
df_results.index.name  = 'implement_order'

# Sprint assignment
def suggest_sprint(rank, total):
    pct = rank / total
    if pct <= 0.25:   return 'Sprint 1'
    elif pct <= 0.50: return 'Sprint 2'
    elif pct <= 0.75: return 'Sprint 3'
    else:             return 'Sprint 4'

total = len(df_results)
df_results['sprint'] = [
    suggest_sprint(i+1, total) for i in range(total)
]

# Display results
print("\n" + "=" * 85)
print("      AGILE REQUIREMENTS — IMPLEMENTATION ORDER")
print("=" * 85)
print(f"{'Rank':<6} {'Priority':<10} {'Sprint':<12} {'Confidence':<13} Requirement")
print("-" * 85)

for idx, row in df_results.iterrows():
    emoji = {'High':'🔴', 'Medium':'🟡', 'Low':'🟢'}[row['priority']]
    print(f"{idx:<6} {emoji} {row['priority']:<8} "
          f"{row['sprint']:<12} {row['confidence']}%"
          f"{'':6} {row['text'][:50]}...")

print("\n" + "=" * 85)
print("📊 SUMMARY:")
print(f"   🔴 High   : {len(df_results[df_results['priority']=='High']):>3} → Sprint 1  (Do First)")
print(f"   🟡 Medium : {len(df_results[df_results['priority']=='Medium']):>3} → Sprint 2-3")
print(f"   🟢 Low    : {len(df_results[df_results['priority']=='Low']):>3} → Sprint 4  (Do Last)")
print("=" * 85)

Ranking all requirements...

      AGILE REQUIREMENTS — IMPLEMENTATION ORDER
Rank   Priority   Sprint       Confidence    Requirement
-------------------------------------------------------------------------------------
1      🔴 High     Sprint 1     89.37%       As an owner, I want to be sure that USAspending on...
2      🔴 High     Sprint 1     87.81%       As a Developer , I want to ensure that attempts to...
3      🔴 High     Sprint 1     87.24%       As an Owner, I want to reset the environment to on...
4      🔴 High     Sprint 1     85.71%       As an agency user, I want to submit my data elemen...
5      🔴 High     Sprint 1     85.28%       As a Developer , I want to update the FABS sample ...
6      🔴 High     Sprint 1     83.68%       As an agency user, I want the FABS validation rule...
7      🔴 High     Sprint 1     82.13%       As an agency user, I want the FABS validation rule...
8      🔴 High     Sprint 1     81.83%       As a Developer, I want D Files generation requests

In [ ]:
#═══════════════════════════════════════════
# STEP 12: Download final results
#═══════════════════════════════════════════
from google.colab import files

# Save results CSV locally
df_results.to_csv("agile_requirements_ranked_v2.csv")

# Save to Google Drive
df_results.to_csv(
    "/content/drive/MyDrive/agile_requirements_ranked_v2.csv"
)

# Download to your computer
files.download("agile_requirements_ranked_v2.csv")

print("✅ Downloaded: agile_requirements_ranked_v2.csv")
print(f"\n🎯 Research Complete!")
print(f"   {len(df_results)} requirements ranked")
print(f"   from #1 (most important) to #{len(df_results)} (least important)")
print(f"   Sprint assignments included")
print(f"   Confidence scores included")

In [ ]:
#═══════════════════════════════════════════
# OPTIONAL STEP 13: Upload new unlabeled dataset
# Use the already-trained model on new data
#═══════════════════════════════════════════
import pandas as pd
from google.colab import files

print("Upload your new unlabeled CSV file:")
new_uploaded = files.upload()

filename     = list(new_uploaded.keys())[0]
df_unlabeled = pd.read_csv(filename, header=None, names=['text', 'source'])

print(f"\n✅ {filename} loaded: {len(df_unlabeled):,} requirements")
print("\nNow re-run STEP 11 to rank them.")